# LEGO GINN Model Test

This notebook tests a trained LEGO GINN model by:
1. Loading a checkpoint
2. Extracting meshes for different brick types
3. Visualizing the results

**Key fix**: Uses correct bounds per brick type (1x2 vs 1x4 have different sizes)

In [15]:
# Setup
import sys
sys.path.append('..')

import torch
import numpy as np
import glob
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [16]:
# Load checkpoint
from util.checkpointing import load_yaml_and_drop_keys
from util.misc import get_model, get_problem
from models.net_w_partials import NetWithPartials

# Find latest checkpoint
ckpt_patterns = [
    'path/to/dir/cond_wire/**/*-model.pt',
    '../path/to/dir/cond_wire/**/*-model.pt',
]

ckpt_path = None
for pattern in ckpt_patterns:
    candidates = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if candidates:
        ckpt_path = candidates[-1]
        break

if ckpt_path is None:
    raise FileNotFoundError("No checkpoint found!")

print(f"Loading: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location='cpu')
print(f"Checkpoint keys: {list(ckpt.keys())}")

Loading: ../path/to/dir/cond_wire\2026_02_04__19_19_25-pyps8bgh\2026_02_04__19_19_25-pyps8bgh-model.pt
Checkpoint keys: ['state_dict', 'init_params']


C:\Users\chukw\AppData\Local\Temp\ipykernel_4372\1744620360.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location='cpu')


In [17]:
# Extract config and model args (init_params in checkpoint overrides for architecture)
config = ckpt.get('config', {})
init_params = ckpt.get('init_params', {})
model_args = config.get('model', {}) if not init_params else {k: v for k, v in init_params.items() if k != 'layers'}
if init_params and 'layers' in init_params:
    raw = init_params['layers']
    model_args['layers'] = raw[1:-1].copy() if len(raw) >= 2 and raw[-1] == 1 else list(raw)
    model_args.setdefault('nz', init_params.get('nz', 2))
    model_args.setdefault('nx', 3)
    model_args.setdefault('use_tiled_coords', init_params.get('use_tiled_coords', False))
    model_args.setdefault('use_dist_to_edge', init_params.get('use_dist_to_edge', False))
    model_args.setdefault('n_studs_values', init_params.get('n_studs_values') or [2, 4])
    model_args['w0_initial'] = model_args.pop('first_omega_0', init_params.get('w0_initial', 18))
    model_args['w0'] = model_args.pop('hidden_omega_0', init_params.get('w0', 1.0))
    model_args['wire_scale'] = model_args.pop('scale', init_params.get('wire_scale', 6))
    if init_params.get('n_studs_values'):
        config['n_studs_values'] = init_params['n_studs_values']
        config['condition_on_n_studs'] = True

print("Config keys:", list(config.keys())[:15])
print("Model args (sample):", {k: model_args.get(k) for k in ['nz', 'nx', 'layers', 'use_tiled_coords', 'use_dist_to_edge']})

# Key parameters
nz = model_args.get('nz', 2)
nx = model_args.get('nx', 3)
n_studs_values = config.get('n_studs_values', [2, 4])
# Include 1x5 (add 5 if not already present)
if 5 not in n_studs_values:
    n_studs_values = list(n_studs_values) + [5]
condition_on_n_studs = config.get('condition_on_n_studs', False)

print(f"\nnz={nz}, nx={nx}")
print(f"n_studs_values={n_studs_values}")
print(f"condition_on_n_studs={condition_on_n_studs}")

Config keys: ['n_studs_values', 'condition_on_n_studs']
Model args (sample): {'nz': 2, 'nx': 3, 'layers': [256, 256, 256, 256], 'use_tiled_coords': True, 'use_dist_to_edge': True}

nz=2, nx=3
n_studs_values=[2, 4, 5, 8, 12]
condition_on_n_studs=True


In [18]:
# Create model and load weights
# First, inspect state_dict to determine actual model architecture
state_dict = ckpt.get('model', ckpt.get('state_dict', None))
if state_dict:
    print("State dict keys:")
    for k in state_dict.keys():
        print(f"  {k}: {state_dict[k].shape}")

# Detect model type from state_dict keys
has_freqs_scale = any('freqs' in k for k in state_dict.keys())
has_freq_scale = any('freq_scale' in k for k in state_dict.keys())

print(f"\nhas_freqs_scale (legacy): {has_freqs_scale}")
print(f"has_freq_scale (new): {has_freq_scale}")

# Infer layer widths from state_dict BEFORE conversion
# Legacy format: freqs.weight shape is [hidden_dim, input_dim]
layer_widths = []
if has_freqs_scale:
    # Find all freqs.weight keys to get hidden dims
    for i in range(10):  # max 10 layers
        key = f'net.{i}.freqs.weight'
        if key in state_dict:
            hidden_dim = state_dict[key].shape[0]
            layer_widths.append(hidden_dim)
        else:
            break
    print(f"Inferred layer widths from legacy format: {layer_widths}")

# If legacy format, we need to convert state_dict keys for ConditionalWIRE
if has_freqs_scale and not has_freq_scale:
    print("\nConverting legacy WIRE state_dict to new format...")
    new_state_dict = {}
    
    for k, v in state_dict.items():
        if '.freqs.' in k:
            layer_idx = k.split('.')[1]
            param_type = k.split('.')[-1]  # weight or bias
            new_state_dict[f'_freqs_{layer_idx}_{param_type}'] = v
        elif '.scale.' in k:
            layer_idx = k.split('.')[1]
            param_type = k.split('.')[-1]
            new_state_dict[f'_scale_{layer_idx}_{param_type}'] = v
        else:
            new_state_dict[k] = v
    
    # Now combine freqs and scale into freq_scale
    final_state_dict = {}
    layer_indices = set()
    for k in new_state_dict.keys():
        if k.startswith('_freqs_'):
            layer_indices.add(k.split('_')[2])
    
    for k, v in new_state_dict.items():
        if k.startswith('_freqs_') or k.startswith('_scale_'):
            continue
        final_state_dict[k] = v
    
    for layer_idx in sorted(layer_indices, key=int):
        for param_type in ['weight', 'bias']:
            freqs_key = f'_freqs_{layer_idx}_{param_type}'
            scale_key = f'_scale_{layer_idx}_{param_type}'
            if freqs_key in new_state_dict and scale_key in new_state_dict:
                # Concatenate freqs and scale tensors along output dimension
                combined = torch.cat([new_state_dict[freqs_key], new_state_dict[scale_key]], dim=0)
                final_state_dict[f'net.{layer_idx}.freq_scale.{param_type}'] = combined
                print(f"  Combined net.{layer_idx}.freq_scale.{param_type}: {combined.shape}")
    
    state_dict = final_state_dict
    print(f"\nConverted state dict keys: {list(state_dict.keys())}")

# Always use cond_wire since that's what's in this codebase
model_str = 'cond_wire'

# Ensure model_args has required fields  
model_args['model_str'] = model_str
if 'nx' not in model_args:
    model_args['nx'] = nx
if 'nz' not in model_args:
    model_args['nz'] = nz

# Match checkpoint: state_dict first-layer input dim is source of truth (11 = nx+nz+tiled+dist)
init_params = ckpt.get('init_params', {})
w0 = state_dict['net.0.freq_scale.weight'] if 'net.0.freq_scale.weight' in state_dict else state_dict.get('net.0.freqs.weight')
ckpt_input_dim = int(w0.shape[1]) if w0 is not None else None
if ckpt_input_dim is not None:
    if ckpt_input_dim == 11:
        model_args['nz'] = 5
        model_args['use_tiled_coords'] = True
        model_args['use_dist_to_edge'] = True
        model_args['n_studs_values'] = model_args.get('n_studs_values') or [2, 4]
    elif ckpt_input_dim == 8:
        model_args['nz'] = 5
        model_args['use_tiled_coords'] = False
        model_args['use_dist_to_edge'] = False
    else:
        model_args['nz'] = ckpt_input_dim - 3
        model_args['use_tiled_coords'] = False
        model_args['use_dist_to_edge'] = False
    print(f"From state_dict: ckpt_input_dim={ckpt_input_dim} -> nz={model_args['nz']}, tiled={model_args.get('use_tiled_coords')}, dist_to_edge={model_args.get('use_dist_to_edge')}")
if init_params and 'layers' in init_params:
    raw = init_params['layers']
    model_args['layers'] = raw[1:-1].copy() if len(raw) >= 2 and raw[-1] == 1 else list(raw)
    for k in ('w0_initial', 'w0', 'wire_scale'):
        if k in init_params:
            model_args[k] = init_params[k]
if layer_widths and 'layers' not in model_args:
    model_args['layers'] = layer_widths
    print(f"Using inferred layers: {layer_widths}")
elif 'layers' not in model_args:
    model_args['layers'] = [256, 256, 256, 256]

if 'return_density' not in model_args:
    model_args['return_density'] = config.get('nf_is_density', False)

# CRITICAL: omega0/sigma0 are non-trainable hyperparams - must be set on model creation
# get_model uses: w0_initial -> first_omega_0, w0 -> hidden_omega_0, wire_scale -> scale
# Remove any conflicting keys and use correct names for get_model
for key in ['first_omega_0', 'hidden_omega_0', 'scale']:
    model_args.pop(key, None)

if 'w0_initial' not in model_args:
    model_args['w0_initial'] = config.get('w0_initial', config.get('first_omega_0', 30.0))
if 'w0' not in model_args:
    model_args['w0'] = config.get('w0', config.get('hidden_omega_0', 30.0))
if 'wire_scale' not in model_args:
    model_args['wire_scale'] = config.get('wire_scale', config.get('sigma0', 10.0))

# Force architecture to match checkpoint right before get_model (avoids stale 9-dim model)
if ckpt_input_dim is not None:
    if ckpt_input_dim == 11:
        model_args['nz'] = 5
        model_args['use_tiled_coords'] = True
        model_args['use_dist_to_edge'] = True
        model_args['n_studs_values'] = model_args.get('n_studs_values') or [2, 4]
    elif ckpt_input_dim == 8:
        model_args['use_tiled_coords'] = False
        model_args['use_dist_to_edge'] = False
    else:
        model_args['use_tiled_coords'] = False
        model_args['use_dist_to_edge'] = False
print(f"\nFinal model_args: {model_args}")

# Pass a copy of layers so get_model doesn't mutate model_args (avoids wrong dim on re-run)
model_args_for_get_model = {**model_args, 'layers': list(model_args.get('layers', [256, 256, 256, 256]))}
# Force architecture from checkpoint so we never pass wrong use_tiled_coords/use_dist_to_edge
if ckpt_input_dim == 11:
    model_args_for_get_model['nz'] = 5
    model_args_for_get_model['use_tiled_coords'] = True
    model_args_for_get_model['use_dist_to_edge'] = True
    model_args_for_get_model['n_studs_values'] = model_args_for_get_model.get('n_studs_values') or [2, 4]
elif ckpt_input_dim == 8:
    model_args_for_get_model['use_tiled_coords'] = False
    model_args_for_get_model['use_dist_to_edge'] = False
elif ckpt_input_dim is not None:
    model_args_for_get_model['use_tiled_coords'] = False
    model_args_for_get_model['use_dist_to_edge'] = False
model = get_model(**model_args_for_get_model)
model_first_in = model.net[0].freq_scale.weight.shape[1]
ckpt_first_in = state_dict['net.0.freq_scale.weight'].shape[1]
if model_first_in != ckpt_first_in:
    raise RuntimeError(
        f"Model input dim ({model_first_in}) != checkpoint ({ckpt_first_in}). "
        "Re-run cells 3 (config) and 4 (this cell) from the start so use_tiled_coords/use_dist_to_edge match the checkpoint."
    )
print(f"Model type: {type(model).__name__} (input_dim={model_first_in})")

# Debug: check that omega_0 and scale_0 are set on layers
for i, layer in enumerate(model.net):
    if hasattr(layer, 'omega_0'):
        print(f"  Layer {i}: omega_0={layer.omega_0}, scale_0={layer.scale_0}")

# Debug: print model state dict keys for comparison
print("\nModel expects these keys:")
for k, v in model.state_dict().items():
    print(f"  {k}: {v.shape}")

# Load weights
model.load_state_dict(state_dict)
print("\nLoaded model weights successfully!")

model.eval()
model.to(device)

# Create NetWithPartials wrapper
netp = NetWithPartials.create_from_model(model, nz=model_args['nz'], nx=model_args['nx'])
netp.params = {k: v.to(device) for k, v in netp.params.items()}
print("Model ready")

State dict keys:
  net.0.freq_scale.weight: torch.Size([512, 11])
  net.0.freq_scale.bias: torch.Size([512])
  net.1.freq_scale.weight: torch.Size([512, 256])
  net.1.freq_scale.bias: torch.Size([512])
  net.2.freq_scale.weight: torch.Size([512, 256])
  net.2.freq_scale.bias: torch.Size([512])
  net.3.freq_scale.weight: torch.Size([512, 256])
  net.3.freq_scale.bias: torch.Size([512])
  net.4.weight: torch.Size([1, 256])
  net.4.bias: torch.Size([1])

has_freqs_scale (legacy): False
has_freq_scale (new): True
From state_dict: ckpt_input_dim=11 -> nz=5, tiled=True, dist_to_edge=True

Final model_args: {'return_density': False, 'use_legacy_gabor': False, 'use_tiled_coords': True, 'stud_spacing_y': 1.0, 'use_dist_to_edge': True, 'n_studs_values': [2, 4, 5, 8, 12], 'n_norm_col': 2, 'use_hypernet': False, 'c_dim': 3, 'layers': [256, 256, 256, 256], 'nz': 5, 'nx': 3, 'w0_initial': 18, 'w0': 1.0, 'wire_scale': 6, 'model_str': 'cond_wire'}
Model type: ConditionalWIRE (input_dim=11)
  Layer 0: 

In [19]:
# Create problems for each brick type - CRITICAL for correct bounds
problems = {}
base_problem_cfg = config.get('problem', {'problem_str': 'lego_1xN', 'n_studs': 4})

# Get sampling config - provide defaults if missing
problem_sampling = config.get('problem_sampling', {})
default_sampling = {
    'nx': 3,
    'n_points_envelope': 1000,
    'n_points_interfaces': 1000, 
    'n_points_domain': 1000,
}
for k, v in default_sampling.items():
    if k not in problem_sampling:
        problem_sampling[k] = v

print(f"Problem sampling: {problem_sampling}")

for n in n_studs_values:
    prob_cfg = {**base_problem_cfg, 'n_studs': n, 'n_studs_y': None, 'height_scale': 1.0}
    # Remove 'problem_str' from sampling if it exists
    sampling = {k: v for k, v in problem_sampling.items() if k != 'problem_str'}
    prob = get_problem(problem_config=prob_cfg, **sampling)
    problems[n] = prob
    print(f"\n1x{n} brick bounds:")
    print(f"  x: [{prob.bounds[0,0]:.3f}, {prob.bounds[0,1]:.3f}]")
    print(f"  y: [{prob.bounds[1,0]:.3f}, {prob.bounds[1,1]:.3f}]")
    print(f"  z: [{prob.bounds[2,0]:.3f}, {prob.bounds[2,1]:.3f}]")

Problem sampling: {'nx': 3, 'n_points_envelope': 1000, 'n_points_interfaces': 1000, 'n_points_domain': 1000}
Created LEGO 1x2 problem (height_scale=1.00) [no_studs=cuboid only]:
  Bounds: [[-0.987500011920929, 0.987500011920929], [-0.48750001192092896, 0.48750001192092896], [-0.6000000238418579, 0.6000000238418579]]
  Stud centers: 0 studs
  Points: 1000 far_outside, 2000 outside, 1000 around_if, 2000 inside, 0 interface, 6 walls

1x2 brick bounds:
  x: [-0.988, 0.988]
  y: [-0.488, 0.488]
  z: [-0.600, 0.600]
Created LEGO 1x4 problem (height_scale=1.00) [no_studs=cuboid only]:
  Bounds: [[-1.9874999523162842, 1.9874999523162842], [-0.48750001192092896, 0.48750001192092896], [-0.6000000238418579, 0.6000000238418579]]
  Stud centers: 0 studs
  Points: 1000 far_outside, 2000 outside, 1000 around_if, 2000 inside, 0 interface, 6 walls

1x4 brick bounds:
  x: [-1.987, 1.987]
  y: [-0.488, 0.488]
  z: [-0.600, 0.600]
Created LEGO 1x5 problem (height_scale=1.00) [no_studs=cuboid only]:
  Boun

In [20]:
# Helper: Create latent vector with conditioning
def make_latent(z_base, n_studs, config):
    """
    Create full latent vector with conditioning.
    
    Args:
        z_base: base latent (2D tensor or list)
        n_studs: number of studs (e.g., 2 for 1x2)
        config: full config dict
    """
    if isinstance(z_base, list):
        z_base = torch.tensor(z_base, dtype=torch.float32)
    z_base = z_base.to(device)
    
    n_vals = config.get('n_studs_values', [2, 2])
    n_min, n_max = min(n_vals), max(n_vals)
    
    # Build conditioning columns
    cond = []
    if config.get('condition_on_n_studs', False):
        n_norm = (n_studs - n_min) / max(n_max - n_min, 1)
        cond.append(n_norm)
    if config.get('condition_on_n_studs_y', False):
        cond.append(0.0)  # ny=1 → normalized to 0
    if config.get('condition_on_height', False):
        cond.append(0.0)  # h=1.0 → normalized to 0 if only one height
    
    if cond:
        cond_tensor = torch.tensor(cond, dtype=torch.float32, device=device)
        z_full = torch.cat([z_base, cond_tensor])
    else:
        z_full = z_base
    
    return z_full

# Test
z_test = make_latent([0.05, 0.05], n_studs=2, config=config)
print(f"Latent for 1x2: {z_test}")
z_test = make_latent([0.05, 0.05], n_studs=2, config=config)
print(f"Latent for 1x4: {z_test}")

Latent for 1x2: tensor([0.0500, 0.0500, 0.0000], device='cuda:0')
Latent for 1x4: tensor([0.0500, 0.0500, 0.0000], device='cuda:0')


In [21]:
# Extract meshes
from util.visualization.utils_mesh import get_watertight_mesh_for_latent

mc_resolution = 128
z_base = [0.05, 0.05]  # Sample latent

# Sanity check: model (netp) must expect same input dim as checkpoint; otherwise re-run cells 1–4
try:
    _n = n_studs_values[0]
    _z = make_latent(z_base, _n, config)
    _x = torch.zeros(1, 3, device=device)
    _ = netp.f_(netp.params, _x, _z.unsqueeze(0))
except RuntimeError as e:
    if "shapes cannot be multiplied" in str(e):
        ckpt_in = netp.params.get("net.0.freq_scale.weight")
        ckpt_in = ckpt_in.shape[1] if ckpt_in is not None else "?"
        raise RuntimeError(
            f"Model input dimension mismatch (checkpoint expects {ckpt_in}-dim input). "
            "Re-run cells 1–4 in order so the model is built with the same architecture as the checkpoint."
        ) from e
    raise

meshes = {}
for n_studs in n_studs_values:
    z = make_latent(z_base, n_studs, config)
    bounds = problems[n_studs].bounds.to(device)
    
    print(f"Extracting 1x{n_studs} mesh with bounds {bounds.tolist()}...")
    verts, faces = get_watertight_mesh_for_latent(
        netp.f_, netp.params, z, bounds, 
        mc_resolution=mc_resolution, 
        device=device,
        chunks=1, 
        level=0, 
        surpress_watertight=True
    )
    meshes[n_studs] = (verts, faces)
    print(f"  Got {len(verts)} vertices, {len(faces)} faces")

RuntimeError: Model input dimension mismatch (checkpoint expects 11-dim input). Re-run cells 1–4 in order so the model is built with the same architecture as the checkpoint.

In [ ]:
# Visualize with k3d
import k3d

fig = k3d.plot(height=600)

colors = [0x3498db, 0xe74c3c, 0x2ecc71, 0xf39c12]
offset_x = 0

for i, n_studs in enumerate(n_studs_values):
    verts, faces = meshes[n_studs]
    if len(verts) > 0:
        # Offset for side-by-side display
        verts_display = np.array(verts, dtype=np.float32)
        verts_display[:, 0] += offset_x
        
        fig += k3d.mesh(
            verts_display, 
            np.array(faces, dtype=np.uint32),
            color=colors[i % len(colors)],
            side='double',
            name=f"1x{n_studs}"
        )
        
        # Update offset based on brick width
        offset_x += problems[n_studs].bounds[0, 1].item() * 2.5
    else:
        print(f"WARNING: Empty mesh for 1x{n_studs}")

fig.display()

Output()

In [ ]:
# Test multiple latent vectors for one brick type
n_studs = n_studs_values[0]
bounds = problems[n_studs].bounds.to(device)

# Grid of latents
grid_size = 3
z_range = np.linspace(0.0, 0.1, grid_size)

fig2 = k3d.plot(height=600)
spacing = 3.0

for i, z0 in enumerate(z_range):
    for j, z1 in enumerate(z_range):
        z = make_latent([z0, z1], n_studs, config)
        verts, faces = get_watertight_mesh_for_latent(
            netp.f_, netp.params, z, bounds,
            mc_resolution=64, device=device,
            chunks=1, level=0, surpress_watertight=True
        )
        
        if len(verts) > 0:
            verts_display = np.array(verts, dtype=np.float32)
            verts_display[:, 0] += j * spacing
            verts_display[:, 1] += i * spacing
            
            fig2 += k3d.mesh(
                verts_display,
                np.array(faces, dtype=np.uint32),
                color=0x4488ff,
                side='double',
                name=f"z=[{z0:.2f},{z1:.2f}]"
            )

print(f"Latent grid for 1x{n_studs}:")
fig2.display()

Latent grid for 1x1:


Output()

### Full layout of the latent space (no filters)
All brick types, full (z0, z1) grid. Each horizontal block is one brick type; within each block, row = z0, column = z1.

In [ ]:
# Full latent space: all brick types, full (z0, z1) grid — no filters
grid_size_full = 5
z_min, z_max = 0.0, 0.15
z_range_full = np.linspace(z_min, z_max, grid_size_full)
spacing = 3.0
block_spacing = 8.0  # gap between brick-type blocks

fig_full = k3d.plot(height=800)

for i_n, n_studs in enumerate(n_studs_values):
    bounds = problems[n_studs].bounds.to(device)
    row_offset = i_n * (grid_size_full * spacing + block_spacing)
    for i, z0 in enumerate(z_range_full):
        for j, z1 in enumerate(z_range_full):
            z = make_latent([z0, z1], n_studs, config)
            verts, faces = get_watertight_mesh_for_latent(
                netp.f_, netp.params, z, bounds,
                mc_resolution=48, device=device,
                chunks=1, level=0, surpress_watertight=True
            )
            if len(verts) > 0:
                verts_display = np.array(verts, dtype=np.float32)
                verts_display[:, 0] += j * spacing
                verts_display[:, 1] += i * spacing + row_offset
                fig_full += k3d.mesh(
                    verts_display,
                    np.array(faces, dtype=np.uint32),
                    color=0x4488ff,
                    side='double',
                    name=f"1x{n_studs} z=[{z0:.2f},{z1:.2f}]"
                )

print("Full latent space: all brick types, z0,z1 in [%.2f,%.2f], %dx%d per type" % (z_min, z_max, grid_size_full, grid_size_full))
print("Rows (bottom→top): 1x2, 1x4, 1x5, 1x8, 1x12. Within each row: z0 (↑), z1 (→)")
fig_full.display()

In [ ]:
# Mesh statistics
import trimesh

print("Mesh Statistics:")
print("=" * 50)

for n_studs in n_studs_values:
    verts, faces = meshes[n_studs]
    if len(verts) == 0:
        print(f"1x{n_studs}: EMPTY MESH")
        continue
    
    tm = trimesh.Trimesh(vertices=verts, faces=faces)
    n_components = len(tm.split(only_watertight=False))
    
    print(f"\n1x{n_studs}:")
    print(f"  Vertices: {len(verts)}")
    print(f"  Faces: {len(faces)}")
    print(f"  Components: {n_components} {'✓' if n_components == 1 else '✗'}")
    print(f"  Watertight: {tm.is_watertight}")
    print(f"  Volume: {tm.volume:.4f}")

Mesh Statistics:

1x1:
  Vertices: 2172433
  Faces: 4874190
  Components: 3073 ✗
  Watertight: False
  Volume: -0.5800

1x4:
  Vertices: 239214
  Faces: 533332
  Components: 361 ✗
  Watertight: True
  Volume: -2.3076

1x5:
  Vertices: 152786
  Faces: 340896
  Components: 209 ✗
  Watertight: True
  Volume: -2.8791
